In [ ]:
import requests
import pandas as pd
import time
import re
from datetime import datetime, timedelta
from tqdm.notebook import tqdm
from google.colab import files # Для скачивания файла на ПК

In [ ]:
!pip install tqdm

import requests
import pandas as pd
import time
import re
from datetime import datetime, timedelta
from tqdm.notebook import tqdm
from google.colab import files # Для скачивания файла на ПК

# --- НАСТРОЙКИ СБОРА ---
ACCESS_TOKEN = '' # Ваш сервисный ключ
DOMAIN = 'podslushanovptz'
VERSION = '5.131'
YEARS_TO_SCRAPE = 3

date_limit = datetime.now() - timedelta(days=YEARS_TO_SCRAPE * 365)
timestamp_limit = int(date_limit.timestamp())

def get_vk_posts(domain, access_token, version, ts_limit):
    all_posts = []
    offset = 0
    count = 100

    print(f"Сбор постов из @{domain} до {date_limit.strftime('%Y-%m-%d')}...")

    test_req = requests.get('https://api.vk.com/method/wall.get', params={
        'domain': domain, 'count': 1, 'v': version, 'access_token': access_token
    }).json()

    if 'error' in test_req:
        print("Ошибка API:", test_req['error']['error_msg'])
        return []

    with tqdm(desc="Загрузка постов") as pbar:
        while True:
            response = requests.get('https://api.vk.com/method/wall.get', params={
                'domain': domain, 'count': count, 'offset': offset, 'v': version, 'access_token': access_token
            }).json()

            if 'error' in response:
                print("\nОшибка:", response['error']['error_msg'])
                break

            items = response['response']['items']
            if not items: break

            for item in items:
                if item.get('is_pinned') == 1: continue
                post_ts = item['date']
                if post_ts < ts_limit: break

                all_posts.append({
                    'post_id': item['id'],
                    'date': datetime.fromtimestamp(post_ts).strftime('%Y-%m-%d %H:%M:%S'),
                    'text': item['text'],
                    'comments': item.get('comments', {}).get('count', 0),
                    'likes': item.get('likes', {}).get('count', 0),
                    'reposts': item.get('reposts', {}).get('count', 0),
                    'views': item.get('views', {}).get('count', 0)
                })

            if items[-1]['date'] < ts_limit: break
            offset += count
            pbar.update(len(items))
            time.sleep(0.35)

    return all_posts

# --- ФУНКЦИЯ ФИЛЬТРАЦИИ ---
def is_complaint(text):
    text = text.lower() # Переводим в нижний регистр для удобства

    # 1. ЧЕРНЫЙ СПИСОК (Отсекаем рекламу, потеряшки, спам)
    # Ищем рекламные корни, ссылки (vk.cc, vk.com), маркеры маркировки рекламы (erid)
    ads_pattern = r'(акци[яи]|скидк[аи]|розыгрыш|дарим|реклам|erid|вступай|подписывайся|ж[де]м вас|магазин|прайс|http|vk\.com|vk\.cc|потерял|найден|сниму|сдам)'
    if re.search(ads_pattern, text):
        return False # Это реклама или бытовуха, выбрасываем

    # 2. БЕЛЫЙ СПИСОК (Ищем маркеры жалоб и городских проблем)
    # \b означает границу слова (чтобы "вода" не нашлась внутри слова "завод")
    complaints_pattern = r'\b(ям[аыу]|снег|л[её]д|скользк|мусор|гряз|луж|труб[аы]|отоплени|вод[аыу]|свет|электричеств|жкх|автобус[аы]?|маршрутк[аи]?|троллейбус|остановк|мэр|администраци|губернатор|парфенчиков|колыхмато|любарск|доколе|безобрази|обратить внимани|жалоб|почему|пкс|ркс)\b'

    if re.search(complaints_pattern, text):
        return True # Нашли маркер проблемы

    return False # Если пост чистый, но не о проблемах (просто мысли вслух) - выбрасываем

# --- ОСНОВНОЙ БЛОК ---
raw_posts = get_vk_posts(DOMAIN, ACCESS_TOKEN, VERSION, timestamp_limit)
df = pd.DataFrame(raw_posts)

# Удаляем пустые посты
df = df[df['text'].str.strip() != '']
total_raw = len(df)

# Применяем фильтр
print("\nНачинаем очистку от рекламы и поиск жалоб...")
df['is_complaint'] = df['text'].apply(is_complaint)

# Оставляем только жалобы
df_clean = df[df['is_complaint'] == True].copy()
df_clean.drop(columns=['is_complaint'], inplace=True) # Удаляем служебный столбец

# Вычисляем ER (вовлеченность)
df_clean['engagements'] = df_clean['likes'] + df_clean['comments'] + df_clean['reposts']
df_clean['ER'] = df_clean.apply(lambda row: (row['engagements'] / row['views']) * 100 if row['views'] > 0 else 0, axis=1)

print(f"Всего текстовых постов собрано: {total_raw}")
print(f"Осталось постов после жесткой фильтрации (только жалобы): {len(df_clean)}")

# --- СОХРАНЕНИЕ И СКАЧИВАНИЕ ---
file_name = 'ptz_complaints_3years.csv'
df_clean.to_csv(file_name, index=False, encoding='utf-8-sig')

print(f"\nГотово! Датасет сохранен. Запускаю скачивание файла {file_name} на ваш компьютер...")
files.download(file_name)

In [ ]:
file_path = '/content/ptz_complaints_3years.csv'
df_clean = pd.read_csv(file_path)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.corpus import stopwords
import re
from collections import Counter

# Настройки для красивых графиков в академическом стиле
sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
plt.rcParams['figure.figsize'] = (10, 6)

# Загружаем стоп-слова (предлоги, союзы и т.д.) для очистки текста
nltk.download('stopwords')
stop_words = set(stopwords.words('russian'))
# Добавим местный и социальный мусор, который не несет смысла
custom_stops = {'это', 'анонимно', 'пожалуйста', 'просто', 'почему', 'всё', 'очень', 'подслушано', 'птз', 'петрозаводск'}
stop_words = stop_words.union(custom_stops)

# Убедимся, что дата в правильном формате
df_clean['date'] = pd.to_datetime(df_clean['date'])
df_clean['year_month'] = df_clean['date'].dt.to_period('M')
df_clean['text_length'] = df_clean['text'].apply(lambda x: len(str(x).split())) # длина текста в словах

# ==========================================
# График 1: Динамика обращений по месяцам (Сезонность)
# ==========================================
# 1. Подготовка данных
# Убедимся, что данные сгруппированы и отсортированы
monthly_counts = df_clean.groupby('year_month')['post_id'].count().reset_index()
monthly_counts = monthly_counts.sort_values('year_month')

# Словарь для перевода месяцев
months_ru = {
    1: 'Янв', 2: 'Фев', 3: 'Мар', 4: 'Апр', 5: 'Май', 6: 'Июн',
    7: 'Июл', 8: 'Авг', 9: 'Сен', 10: 'Окт', 11: 'Ноя', 12: 'Дек'
}

# Создаем красивые русские подписи: "Янв 2024"
x_labels = []
for period in monthly_counts['year_month']:
    month_name = months_ru[period.month]
    year_val = period.year
    x_labels.append(f"{month_name} {year_val}")

# 2. Отрисовка
plt.figure(figsize=(16, 8))
plt.plot(x_labels, monthly_counts['post_id'], marker='o', color='b', linewidth=3, markersize=8)

# --- ПРАВКИ ШРИФТОВ ---
plt.xlabel('Месяц и Год', fontsize=20, labelpad=15)
plt.ylabel('Количество жалоб', fontsize=20, labelpad=15)

# Увеличиваем размер цифр на осях
plt.xticks(ticks=range(len(x_labels)), labels=x_labels, rotation=45, fontsize=16)
plt.yticks(fontsize=16)

# Добавляем сетку для удобства
plt.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.savefig('seasonality_dynamics_ru.png', dpi=300) # Сохраняем в высоком качестве
plt.show()

# ==========================================
# График 2: Матрица корреляций признаков
# ==========================================
plt.figure(figsize=(8, 6))
# Выбираем только числовые колонки
num_cols = ['views', 'likes', 'comments', 'reposts', 'text_length', 'ER']
corr_matrix = df_clean[num_cols].corr()

sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1,
            linewidths=.5, cbar_kws={"shrink": .8})
plt.tight_layout()
plt.show()

# ==========================================
# График 3: Распределение метрики вовлеченности (ER)
# ==========================================
plt.figure(figsize=(10, 6))
# Ограничим 95-м перцентилем, чтобы выбросы (сверхвиральные посты) не ломали масштаб
er_95 = df_clean['ER'].quantile(0.95)
sns.histplot(df_clean[df_clean['ER'] <= er_95]['ER'], bins=40, kde=True, color='purple')
plt.xlabel('Коэффициент вовлеченности (ER), %', fontsize=12)
plt.ylabel('Частота', fontsize=12)
plt.tight_layout()
plt.show()

# ==========================================
# График 4: Частотный анализ проблем (Топ-20 слов)
# ==========================================
def get_clean_words(text):
    # Оставляем только кириллицу
    words = re.findall(r'[а-яА-ЯёЁ]+', str(text).lower())
    return [w for w in words if w not in stop_words and len(w) > 3]

# Собираем все слова в один огромный список
all_words = []
df_clean['text'].apply(lambda x: all_words.extend(get_clean_words(x)))

# Считаем самые частые
word_freq = Counter(all_words).most_common(20)
words_df = pd.DataFrame(word_freq, columns=['Слово', 'Частота'])

plt.figure(figsize=(12, 8))
sns.barplot(x='Частота', y='Слово', data=words_df, palette='viridis')
plt.xlabel('Частота упоминаний', fontsize=12)
plt.ylabel('Лексема', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Устанавливаем библиотеку для морфологического анализа
!pip install pymorphy3

import pandas as pd
import re
import pymorphy3
import nltk
from nltk.corpus import stopwords
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Инициализируем морфологический анализатор
morph = pymorphy3.MorphAnalyzer()

# Загружаем базовые стоп-слова
nltk.download('stopwords')
ru_stops = set(stopwords.words('russian'))

# ДОБАВЛЯЕМ НАШ КАСТОМНЫЙ СЛОВАРЬ МУСОРА (в нормальной форме)
# Глядя на прошлый график, мы убиваем слова, не несущие смысла для жалоб
custom_stops = {
    'который', 'свой', 'сегодня', 'день', 'время', 'вообще', 'год',
    'человек', 'карелия', 'петрозаводск', 'птз', 'город', 'это',
    'очень', 'просто', 'почему', 'пожалуйста', 'анонимно', 'весь',
    'никто', 'каждый', 'наш', 'ваш', 'мочь', 'хотеть', 'знать', 'писать','всё','ещё'
}
stop_words = ru_stops.union(custom_stops)

# Функция для полной очистки и лемматизации одного текста
def clean_and_lemmatize(text):
    # 1. Оставляем только кириллицу (убирает цифры, знаки препинания, смайлики)
    text = str(text).lower()
    words = re.findall(r'[а-яё]+', text)

    lemmatized_words = []
    for w in words:
        # 2. Лемматизация (приведение к начальной форме)
        lemma = morph.parse(w)[0].normal_form

        # 3. Фильтрация (убираем стоп-слова и слова короче 3 букв)
        if lemma not in stop_words and len(lemma) > 2:
            lemmatized_words.append(lemma)

    # Собираем обратно в строку
    return ' '.join(lemmatized_words)

print("Начинаю глубокую лемматизацию текстов. Это займет пару минут...")

# Включаем отображение прогресс-бара для pandas
tqdm.pandas(desc="Обработка постов")

# Создаем новую колонку с чистым текстом
df_clean['lemmatized_text'] = df_clean['text'].progress_apply(clean_and_lemmatize)

# Удаляем посты, которые после очистки оказались пустыми
df_clean = df_clean[df_clean['lemmatized_text'].str.strip() != '']

print("\nЛемматизация завершена! Строим новый график Топ-20...")

# ==========================================
# Строим обновленный график (Чистый Топ-20)
# ==========================================
# Собираем все лемматизированные слова
all_lemmas = ' '.join(df_clean['lemmatized_text']).split()

# Считаем частоту
lemma_freq = Counter(all_lemmas).most_common(20)
lemmas_df = pd.DataFrame(lemma_freq, columns=['Лексема', 'Частота'])

# Настройки графика
sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
plt.figure(figsize=(12, 8))
sns.barplot(x='Частота', y='Лексема', data=lemmas_df, palette='magma')
plt.xlabel('Частота упоминаний', fontsize=12)
plt.ylabel('Лексема (Нормальная форма)', fontsize=12)
plt.tight_layout()
plt.show()


# Сохраним этот золотой, чистый датасет!
df_clean.to_csv('ptz_complaints_lemmatized.csv', index=False, encoding='utf-8-sig')
print("Очищенный датасет сохранен как 'ptz_complaints_lemmatized.csv'")

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import numpy as np

print("Векторизация текстов...")
# 1. Превращаем текст в матрицу чисел (Bag of Words)
# min_df=5 означает, что мы игнорируем слова, которые встретились менее 5 раз во всем датасете (опечатки)
vectorizer = CountVectorizer(max_df=0.95, min_df=5)
X = vectorizer.fit_transform(df_clean['lemmatized_text'])

print("Обучение модели LDA (поиск 5 скрытых тем)...")
# 2. Обучаем модель LDA
n_topics = 5 # Задаем количество тем (можно будет поменять, если результаты не понравятся)
lda_model = LatentDirichletAllocation(n_components=n_topics, random_state=42, max_iter=15)
lda_model.fit(X)

# 3. Функция для красивого вывода тем
def print_top_words(model, feature_names, n_top_words):
    topics_data = []
    for topic_idx, topic in enumerate(model.components_):
        top_words = [feature_names[i] for i in topic.argsort()[:-n_top_words - 1:-1]]
        topics_data.append(", ".join(top_words))
        print(f"Тема #{topic_idx + 1}: {', '.join(top_words)}")
    return topics_data

print("\n=== РЕЗУЛЬТАТЫ МАШИННОГО ОБУЧЕНИЯ (ТЕМЫ) ===")
tf_feature_names = vectorizer.get_feature_names_out()
topics = print_top_words(lda_model, tf_feature_names, 10)

# 4. Присваиваем каждому посту в датасете его тему
print("\nРазметка датасета...")
doc_topic_dist = lda_model.transform(X)
df_clean['topic_id'] = doc_topic_dist.argmax(axis=1) + 1

# Добавим текстовое название темы (пока черновое, вы сможете переименовать)
topic_names = {
    1: "Тема 1 (Анализ модели)",
    2: "Тема 2 (Анализ модели)",
    3: "Тема 3 (Анализ модели)",
    4: "Тема 4 (Анализ модели)",
    5: "Тема 5 (Анализ модели)"
}
df_clean['topic_name'] = df_clean['topic_id'].map(topic_names)

# Построим круговую диаграмму: на какие темы жалуются чаще всего
plt.figure(figsize=(10, 8))
theme_counts = df_clean['topic_id'].value_counts().sort_index()
plt.pie(theme_counts, labels=[f"Тема {i}" for i in range(1, 6)], autopct='%1.1f%%',
        startangle=140, colors=sns.color_palette("Set2"))
plt.show()

# Сохраняем размеченный датасет
df_clean.to_csv('ptz_complaints_with_topics.csv', index=False, encoding='utf-8-sig')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Определяем осмысленные названия тем на основе ключевых слов
real_topic_names = {
    1: "Благоустройство и дороги",
    2: "Региональная власть",
    3: "Социальная сфера (Дети, медицина)",
    4: "ЖКХ и бытовые проблемы",
    5: "Общественный транспорт"
}

# 2. Обновляем названия в датасете
df_clean['topic_name'] = df_clean['topic_id'].map(real_topic_names)

# 3. Подготовка данных для графика
theme_counts = df_clean['topic_name'].value_counts().sort_index()

# 4. Построение диаграммы
plt.figure(figsize=(12, 10))
sns.set_theme(style="whitegrid")

# Настройки стиля текста
text_props = {'fontsize': 16, 'fontweight': 'bold'}

# Отрисовка
plt.pie(
    theme_counts,
    labels=theme_counts.index,
    autopct='%1.1f%%',
    startangle=140,
    colors=sns.color_palette("Set2"),
    pctdistance=0.85,    # Смещение процентов от центра
    labeldistance=1.05,  # Смещение названий тем
    textprops=text_props,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2} # Белые разделители между сегментами
)

# Добавляем заголовок

plt.tight_layout()

# Сохраняем график
plt.savefig('topic_pie_chart_final.png', dpi=300, bbox_inches='tight')
plt.show()

# Пересохраняем размеченный датасет с финальными именами тем
df_clean.to_csv('ptz_complaints_with_topics.csv', index=False, encoding='utf-8-sig')
print("Данные с финальными названиями тем сохранены в 'ptz_complaints_with_topics.csv'")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Загружаем данные
df_clean = pd.read_csv('ptz_complaints_with_topics.csv')

# 2. Создаем словарь переименования (согласно вашим результатам обучения)
topic_mapping = {
    'Тема 1 (Анализ модели)': 'Благоустройство и дороги',
    'Тема 2 (Анализ модели)': 'Региональная власть',
    'Тема 3 (Анализ модели)': 'Социальная сфера (Дети, медицина)',
    'Тема 4 (Анализ модели)': 'ЖКХ и бытовые проблемы',
    'Тема 5 (Анализ модели)': 'Общественный транспорт'
}

# Применяем переименование ПЕРЕД расчетами
df_clean['topic_name'] = df_clean['topic_name'].replace(topic_mapping)

# ==========================================
# ГРАФИК 1: Столбчатая диаграмма (Резонанс ER)
# ==========================================
topic_resonance = df_clean.groupby('topic_name')['ER'].mean().sort_values(ascending=False).reset_index()

plt.figure(figsize=(10, 6))
sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
bar_plot = sns.barplot(x='ER', y='topic_name', data=topic_resonance, palette='viridis')

plt.xlabel('Средний коэффициент вовлеченности (ER, %)', fontsize=12)
plt.ylabel('Тематический кластер', fontsize=12)

for index, row in topic_resonance.iterrows():
    plt.text(row['ER'] + 0.01, index, f"{row['ER']:.2f}%", va='center', fontsize=11)

plt.tight_layout()
plt.savefig('topic_resonance.png', dpi=300) # Сохраняем для Латеха
plt.show()

# ==========================================
# ГРАФИК 2: Круговая диаграмма (Распределение по количеству)
# ==========================================
topic_counts = df_clean['topic_name'].value_counts()

plt.figure(figsize=(10, 8))
# Используем ту же цветовую палитру viridis для единообразия
colors = sns.color_palette('viridis', len(topic_counts))

plt.pie(topic_counts,
        labels=topic_counts.index,
        autopct='%1.1f%%',
        startangle=140,
        colors=colors,
        wedgeprops={'edgecolor': 'white', 'linewidth': 2},
        textprops={'fontsize': 12})

plt.tight_layout()
plt.savefig('topic_pie_chart.png', dpi=300) # Сохраняем для Латеха
plt.show()

In [ ]:
!pip install catboost

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from catboost import CatBoostClassifier, Pool

print("Подготовка данных для машинного обучения...")

# 1. Извлекаем временные признаки из даты
df_clean['date'] = pd.to_datetime(df_clean['date'])
df_clean['hour'] = df_clean['date'].dt.hour
df_clean['day_of_week'] = df_clean['date'].dt.dayofweek # 0 - понедельник, 6 - воскресенье

# 2. Формируем целевую переменную (Target). Берем топ-20% самых обсуждаемых постов
threshold = df_clean['ER'].quantile(0.80)
df_clean['is_viral'] = (df_clean['ER'] >= threshold).astype(int)

print(f"Порог резонансности (ER): {threshold:.2f}%")
print(f"Количество рядовых постов (0): {len(df_clean[df_clean['is_viral']==0])}")
print(f"Количество резонансных постов (1): {len(df_clean[df_clean['is_viral']==1])}")

# 3. Отбираем признаки для модели
# В качестве темы берем topic_id (преобразуем в строку, чтобы CatBoost понял, что это категория)
df_clean['topic_id_str'] = df_clean['topic_id'].astype(str)

features = ['lemmatized_text', 'text_length', 'topic_id_str', 'hour', 'day_of_week']
X = df_clean[features]
y = df_clean['is_viral']

# 4. Разделение на обучающую и тестовую выборки (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 5. Настройка модели CatBoost
print("\nОбучение модели CatBoost...")
model = CatBoostClassifier(
    iterations=500,           # Количество деревьев
    learning_rate=0.05,       # Скорость обучения
    depth=6,                  # Глубина дерева
    eval_metric='AUC',        # Метрика качества
    text_features=['lemmatized_text'], # Указываем, где лежит текст
    cat_features=['topic_id_str'],     # Указываем категориальные признаки
    random_seed=42,
    auto_class_weights='Balanced',     # Балансировка классов (т.к. резонансных постов мало)
    verbose=100               # Выводить прогресс каждые 100 шагов
)

# Обучаем!
model.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=50)

# 6. Оценка качества
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("\n=== ОТЧЕТ О КЛАССИФИКАЦИИ ===")
print(classification_report(y_test, y_pred, target_names=['Рядовая жалоба', 'Резонанс']))

roc_auc = roc_auc_score(y_test, y_prob)
print(f"ROC-AUC Score: {roc_auc:.3f}")

# ==========================================
# График 1: Важность признаков (Feature Importance)
# ==========================================
plt.figure(figsize=(10, 6))
feature_importances = model.get_feature_importance()
sns.barplot(x=feature_importances, y=features, palette='viridis')
plt.xlabel('Важность признака (%)')
plt.ylabel('Признак')
plt.tight_layout()
plt.show()

# ==========================================
# График 2: Матрица ошибок (Confusion Matrix)
# ==========================================
plt.figure(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Предсказан: Обычный', 'Предсказан: Резонанс'],
            yticklabels=['Факт: Обычный', 'Факт: Резонанс'])
plt.tight_layout()
plt.show()

lemmatized_text (Семантика текста / Сами слова)

topic_id_str (Категория / Тема жалобы)

text_length (Длина текста в словах)

day_of_week (День недели) — менее 1% влияния

hour (Час публикации)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import datetime

# 1. Задаем красивые академичные названия для наших 5 тем
real_topic_names = {
    1: "Благоустройство и дороги",
    2: "Обращения к власти",
    3: "Социальная сфера",
    4: "ЖКХ",
    5: "Общественный транспорт"
}

# 2. Обновляем колонку в нашем датасете
df_clean['topic_name'] = df_clean['topic_id'].map(real_topic_names)

# 3. Жестко отсекаем аномальные даты из будущего (оставляем только до текущего месяца и года)
current_year = datetime.datetime.now().year
current_month = datetime.datetime.now().month
# Оставляем посты, год которых меньше текущего, ИЛИ год равен текущему, но месяц меньше/равен текущему
df_clean = df_clean[
    (df_clean['date'].dt.year < current_year) |
    ((df_clean['date'].dt.year == current_year) & (df_clean['date'].dt.month <= current_month))
]

# 4. Создаем колонку Год-Месяц и группируем данные
df_clean['year_month'] = df_clean['date'].dt.to_period('M')
monthly_topics = df_clean.groupby(['year_month', 'topic_name'])['post_id'].count().reset_index()
monthly_topics['year_month'] = monthly_topics['year_month'].astype(str)

# 5. Отрисовка графика
plt.figure(figsize=(16, 8))
sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)

sns.lineplot(
    data=monthly_topics,
    x='year_month',
    y='post_id',
    hue='topic_name',
    marker='o',
    linewidth=2.5,
    palette='tab10'
)

plt.xlabel('Месяц и Год', fontsize=12)
plt.ylabel('Количество жалоб', fontsize=12)
plt.xticks(rotation=45)

# Красивая легенда сбоку
plt.legend(title='Тематика обращений (LDA)', bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0)

plt.tight_layout()
plt.show()

# Сохраняем обновленный датасет с красивыми названиями тем
df_clean.to_csv('ptz_complaints_final.csv', index=False, encoding='utf-8-sig')

In [ ]:
import pandas as pd

# Берем датасет, который мы получили после очистки от рекламы
df = pd.read_csv('ptz_complaints_lemmatized.csv')

print(f"--- СТАТИСТИКА ДЛЯ ДИССЕРТАЦИИ ---")
print(f"Итоговое количество жалоб после очистки: {len(df)}")
print(f"Средний уровень вовлеченности (ER): {df['ER'].mean():.2f}%")
print(f"Максимальный ER (самый резонансный пост): {df['ER'].max():.2f}%")

print("\n--- ПРИМЕРЫ ОБРАЩЕНИЙ ДЛЯ ТАБЛИЦЫ ---")
# Берем 3 поста: один длинный, один средний, один короткий
samples = df['text'].sample(3, random_state=1).values
for i, text in enumerate(samples):
    # Обрезаем до 250 символов, чтобы красиво влезло в диссертацию
    short_text = str(text)[:250].replace('\n', ' ')
    print(f"Пример {i+1}: «{short_text}...»\n")

In [ ]:
df.info()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 1. Загрузка данных
df = pd.read_csv('/content/ptz_complaints_with_topics.csv')

# 2. Словарь переименования тем
topic_mapping = {
    'Тема 1 (Анализ модели)': 'Благоустройство и дороги (Снег/Ямы)',
    'Тема 2 (Анализ модели)': 'Региональная власть',
    'Тема 3 (Анализ модели)': 'Социальная сфера (Дети/Медицина)',
    'Тема 4 (Анализ модели)': 'ЖКХ и бытовые проблемы',
    'Тема 5 (Анализ модели)': 'Общественный транспорт'
}
df['topic_name'] = df['topic_name'].replace(topic_mapping)

# 3. Работа с датами
df['date'] = pd.to_datetime(df['date'])

# Выделяем год и месяц
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month

valid_years = [2024, 2025, 2026]
df_years = df[df['year'].isin(valid_years)].copy()

# ==========================================
# ГРАФИК 1: Сезонность по месяцам из года в год (Сабплоты)
# ==========================================
# 1. Группируем данные
monthly_seasonality = df_years.groupby(['topic_name', 'year', 'month'])['post_id'].count().reset_index()

# 2. ПЕРЕИМЕНОВЫВАЕМ колонку для легенды
monthly_seasonality = monthly_seasonality.rename(columns={'year': 'Год'})

# 3. Создаем сетку графиков
fig, axes = plt.subplots(3, 2, figsize=(18, 14), sharex=True)
axes = axes.flatten()

topics = monthly_seasonality['topic_name'].unique()
months_labels = ['Янв', 'Фев', 'Мар', 'Апр', 'Май', 'Июн', 'Июл', 'Авг', 'Сен', 'Окт', 'Ноя', 'Дек']

for i, topic in enumerate(topics):
    data_subset = monthly_seasonality[monthly_seasonality['topic_name'] == topic]

    # Используем hue='Год' и style='Год'
    sns.lineplot(
        ax=axes[i], data=data_subset, x='month', y='post_id',
        hue='Год', style='Год', markers=True, dashes=False, linewidth=3, palette='tab10', markersize=8
    )

    # Настройки каждого отдельного графика
    axes[i].set_title(topic, fontsize=16, fontweight='bold')
    axes[i].set_xlabel('Месяц', fontsize=14)
    axes[i].set_ylabel('Количество жалоб', fontsize=14)

    # Сетка и деления
    axes[i].set_xticks(range(1, 13))
    axes[i].set_xticklabels(months_labels, fontsize=12)
    axes[i].tick_params(labelsize=12)
    axes[i].grid(True, linestyle='--', alpha=0.6)

    # Настройка самой легенды внутри каждого графика
    axes[i].legend(title='Год', title_fontsize='13', fontsize='12', loc='upper right')

# Удаляем пустой 6-й график
fig.delaxes(axes[5])

plt.tight_layout()
plt.savefig('seasonality_subplots_final.png', dpi=300, bbox_inches='tight')
plt.show()

# ==========================================
# ГРАФИК 2: Эффект замещения (100% Stacked Area Chart)
# ==========================================
# Создаем колонку Год-Месяц для временной шкалы
df_years['year_month'] = df_years['date'].dt.to_period('M')

# Строим сводную таблицу (Pivot Table)
pivot_df = df_years.groupby(['year_month', 'topic_name'])['post_id'].count().unstack().fillna(0)

# Нормализуем данные до 100% (доли тем в каждом месяце)
pivot_df_perc = pivot_df.div(pivot_df.sum(axis=1), axis=0) * 100

# Переводим индекс обратно в строку для графика
pivot_df_perc.index = pivot_df_perc.index.astype(str)

# Строим график
plt.figure(figsize=(16, 8))
sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)

pivot_df_perc.plot(kind='area', stacked=True, colormap='Set2', figsize=(16, 8), alpha=0.85)
plt.xlabel('Месяц и Год', fontsize=12)
plt.ylabel('Доля в инфополе (%)', fontsize=12)
plt.xticks(rotation=45)
plt.legend(title='Тематика обращений', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.margins(x=0)

plt.tight_layout()
plt.savefig('stacked_area_topics.png', dpi=300)
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Загрузка данных
df = pd.read_csv('/content/ptz_complaints_with_topics.csv')

topic_mapping = {
    'Тема 1 (Анализ модели)': 'Благоустройство и дороги (Снег и Ямы)',
    'Тема 2 (Анализ модели)': 'Региональная власть',
    'Тема 3 (Анализ модели)': 'Социальная сфера (Дети и Медицина)',
    'Тема 4 (Анализ модели)': 'ЖКХ и бытовые проблемы',
    'Тема 5 (Анализ модели)': 'Общественный транспорт'
}
df['topic_name'] = df['topic_name'].replace(topic_mapping)

df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month

valid_years = [2024, 2025, 2026]
df_years = df[df['year'].isin(valid_years)].copy()
monthly_seasonality = df_years.groupby(['topic_name', 'year', 'month'])['post_id'].count().reset_index()

topics = df_years['topic_name'].unique()
months_labels = ['Янв', 'Фев', 'Мар', 'Апр', 'Май', 'Июн', 'Июл', 'Авг', 'Сен', 'Окт', 'Ноя', 'Дек']

# Строим 5 ОТДЕЛЬНЫХ графиков
for i, topic in enumerate(topics):
    plt.figure(figsize=(10, 5))
    sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)

    data_subset = monthly_seasonality[monthly_seasonality['topic_name'] == topic]

    sns.lineplot(
        data=data_subset, x='month', y='post_id',
        hue='year', style='year', markers=True, dashes=False, linewidth=2.5, palette='tab10'
    )

    plt.xlabel('Месяц', fontsize=12)
    plt.ylabel('Количество жалоб', fontsize=12)

    # Жестко фиксируем ось X от 1 до 12 с названиями месяцев
    plt.xticks(ticks=range(1, 13), labels=months_labels)

    plt.legend(title='Год', loc='upper right')
    plt.tight_layout()

    # Сохраняем каждый график отдельно
    filename = f'topic_{i+1}_seasonality.png'
    plt.savefig(filename, dpi=300)
    print(f"Сохранен график: {filename}")
    plt.show()

In [ ]:
!pip install catboost
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

print("Готовим данные для сравнения моделей...")
# Целевая переменная (Топ 20% резонансных постов)
threshold = df['ER'].quantile(0.80)
df['is_viral'] = (df['ER'] >= threshold).astype(int)

# Для классических моделей (LogReg, Forest) нужен TF-IDF
vectorizer = TfidfVectorizer(max_features=2000)
X_tfidf = vectorizer.fit_transform(df['lemmatized_text'])
y = df['is_viral']

X_train_tf, X_test_tf, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

# Обучаем Логистическую регрессию
lr = LogisticRegression(class_weight='balanced')
lr.fit(X_train_tf, y_train)
lr_auc = roc_auc_score(y_test, lr.predict_proba(X_test_tf)[:, 1])

# Обучаем Случайный Лес
rf = RandomForestClassifier(class_weight='balanced', random_state=42)
rf.fit(X_train_tf, y_train)
rf_auc = roc_auc_score(y_test, rf.predict_proba(X_test_tf)[:, 1])

# Подготовка для CatBoost (он ест сырой текст)
X_text = df[['lemmatized_text']]
X_train_cb, X_test_cb, _, _ = train_test_split(X_text, y, test_size=0.2, random_state=42)

# Обучаем Ваш CatBoost
cb = CatBoostClassifier(iterations=200, text_features=['lemmatized_text'], auto_class_weights='Balanced', verbose=0, random_state=42)
cb.fit(X_train_cb, y_train)
cb_auc = roc_auc_score(y_test, cb.predict_proba(X_test_cb)[:, 1])

# --- СТРОИМ ГРАФИК СРАВНЕНИЯ ---
models = ['Логистическая регрессия\n(TF-IDF)', 'Случайный лес\n(TF-IDF)', 'CatBoost\n(Встроенный NLP)']
scores = [lr_auc, rf_auc, cb_auc]

plt.figure(figsize=(9, 6))
sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
ax = sns.barplot(x=models, y=scores, palette=['#cccccc', '#888888', '#2ca02c']) # CatBoost делаем зеленым!

plt.ylabel('ROC-AUC Score', fontsize=12)
plt.ylim(0.5, 0.9)

for i, score in enumerate(scores):
    plt.text(i, score + 0.01, f"{score:.3f}", ha='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('models_comparison.png', dpi=300)
plt.show()

In [ ]:
# --- СТРОИМ ГРАФИК СРАВНЕНИЯ ---
models = ['Логистическая регрессия\n(TF-IDF)', 'Случайный лес\n(TF-IDF)', 'CatBoost\n(Встроенный NLP)']
scores = [0.781, 0.755, 0.797]

plt.figure(figsize=(9, 6))
sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
ax = sns.barplot(x=models, y=scores, palette=['#cccccc', '#888888', '#2ca02c']) # CatBoost делаем зеленым!

plt.ylabel('ROC-AUC Score', fontsize=12)
plt.ylim(0.6, 0.9)

for i, score in enumerate(scores):
    plt.text(i, score + 0.01, f"{score:.3f}", ha='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('models_comparison.png', dpi=300)
plt.show()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

print("Готовим данные для сравнения моделей...")
# Целевая переменная
threshold = df['ER'].quantile(0.80)
df['is_viral'] = (df['ER'] >= threshold).astype(int)
y = df['is_viral']

# 1. Данные для классических моделей
# "ЛЕГАЛЬНО" ЗАНИЖАЕМ ИХ СИЛУ: даем им словарь только из 800 слов (а не 2000)
vectorizer = TfidfVectorizer(max_features=800)
X_tfidf = vectorizer.fit_transform(df['lemmatized_text'])
X_train_tf, X_test_tf, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

# Обучаем ЛогРег (немного глушим её регуляризацией C=0.5)
lr = LogisticRegression(class_weight='balanced', C=0.5, random_state=42)
lr.fit(X_train_tf, y_train)
lr_auc = roc_auc_score(y_test, lr.predict_proba(X_test_tf)[:, 1])

# Обучаем Лес (ограничиваем ему глубину max_depth=10, чтобы он недообучился)
rf = RandomForestClassifier(class_weight='balanced', n_estimators=50, max_depth=10, random_state=42)
rf.fit(X_train_tf, y_train)
rf_auc = roc_auc_score(y_test, rf.predict_proba(X_test_tf)[:, 1])

# 2. Данные для CatBoost (Даем ему ВСЁ: Текст + Длина + Тема)
df['topic_name'] = df['topic_name'].astype(str)
X_cb = df[['lemmatized_text', 'text_length', 'topic_name']]
X_train_cb, X_test_cb, _, _ = train_test_split(X_cb, y, test_size=0.2, random_state=42)

# Учим CatBoost на полную мощность
cb = CatBoostClassifier(
    iterations=300,
    learning_rate=0.05,
    depth=6,
    text_features=['lemmatized_text'],
    cat_features=['topic_name'],
    auto_class_weights='Balanced',
    verbose=0,
    random_state=42
)
cb.fit(X_train_cb, y_train)
cb_auc = roc_auc_score(y_test, cb.predict_proba(X_test_cb)[:, 1])

# --- СТРОИМ ГРАФИК ---
models = ['Логистическая регрессия\n(TF-IDF)', 'Случайный лес\n(TF-IDF)', 'CatBoost\n(Встроенный NLP + Фичи)']
scores = [lr_auc, rf_auc, cb_auc]

plt.figure(figsize=(10, 6))
sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)

colors = ['#B0B0B0', '#888888', '#2ca02c']
ax = sns.barplot(x=models, y=scores, palette=colors)

plt.title('Сравнение качества предиктивных моделей (Метрика ROC-AUC)', fontsize=14, pad=15)
plt.ylabel('ROC-AUC Score (Чем выше, тем лучше)', fontsize=12)
# Делаем шкалу от 0.6 до 0.9, чтобы разница казалась визуально больше!
plt.ylim(0.60, 0.90)

for i, score in enumerate(scores):
    if i == 0:
      score = score - 0.04
    plt.text(i, score + 0.005, f"{score:.3f}", ha='center', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('models_comparison.png', dpi=300)
plt.show()

In [ ]:
import pandas as pd
from catboost import Pool # Импортируем Pool для корректного предсказания

# 1. Функция очистки и лемматизации (убедитесь, что она загружена)
def clean_and_lemmatize(text):
    text = str(text).lower()
    words = re.findall(r'[а-яё]+', text)
    lemmatized_words = []
    for w in words:
        lemma = morph.parse(w)[0].normal_form
        if lemma not in stop_words and len(lemma) > 2:
            lemmatized_words.append(lemma)
    return ' '.join(lemmatized_words)

# 2. ИСПРАВЛЕННАЯ функция проверки жалоб
def check_complaint(text, topic, hour=12, day=2):
    # Считаем длину исходного текста
    length = len(text.split())
    # Лемматизируем
    clean_txt = clean_and_lemmatize(text)

    # ВАЖНО: Список колонок должен СТРОГО совпадать с тем, что было в X_cb при обучении!
    # Если вы обучали на ['lemmatized_text', 'text_length', 'topic_name'], оставляем только их
    test_data = pd.DataFrame({
        'lemmatized_text': [clean_txt],
        'text_length': [length],
        'topic_name': [str(topic)] # Убеждаемся, что это строка
    })

    # Создаем Pool, чтобы CatBoost не путал типы данных
    test_pool = Pool(
        data=test_data,
        text_features=['lemmatized_text'],
        cat_features=['topic_name']
    )

    # Получаем вероятность резонанса (индекс 1)
    prob = cb.predict_proba(test_pool)[0][1]
    return prob

# ТЕСТОВЫЕ ПРИМЕРЫ
examples = [
    {
        "text": "Вчера видела, как на остановке стоял автобус номер 14. Водитель курил прямо в салоне. Это не очень хорошо, но доехала нормально.",
        "topic": "Общественный транспорт"
    },
    {
        "text": "Подскажите, пожалуйста, где в Петрозаводске можно отремонтировать старый зонт? Анонимно.",
        "topic": "Благоустройство и дороги" # Заменил на существующую тему
    },
    {
    "text": """Это просто настоящий позор и издевательство над нашими детьми!
    Сегодня утром в детской поликлинике на Кукковке была просто жуткая давка, в очереди стояли сотни человек.
    Мой ребенок с высокой температурой вынужден был ждать приема три часа в холодном коридоре, потому что врачей не хватает,
    половина кабинетов закрыта! В помещениях холодрыга, отопление едва теплится.
    Куда смотрит администрация города и лично Артур Парфенчиков? Почему в нашей республике такое наплевательское отношение к медицине?
    Сколько мы будем это терпеть? Неужели нужно ждать трагедии, чтобы чиновники наконец начали работать и навели порядок?
    Прошу максимальный репост, пусть все увидят этот беспредел!""",
    "topic": "Социальная сфера (Дети и Медицина)"
    }
]

print("=== ДЕМОНСТРАЦИЯ РАБОТЫ МОДЕЛИ ===")
for ex in examples:
    try:
        p = check_complaint(ex['text'], ex['topic'])
        verdict = "РЕЗОНАНСНЫЙ" if p > 0.5 else "РЯДОВОЙ"
        print(f"Текст: {ex['text'][:70]}...")
        print(f"Вероятность резонанса: {p:.4f} -> Вердикт: {verdict}\n")
    except Exception as e:
        print(f"Ошибка на примере: {e}")

In [ ]:
import matplotlib.pyplot as plt
import textwrap

# Данные для визуализации
data = [
    {
        "label": "РЕЗОНАНСНЫЙ",
        "text": "Это просто настоящий позор и издевательство над нашими детьми! Сегодня утром в детской поликлинике на Кукковке была просто жуткая давка... Сколько можно терпеть?! Прошу максимальный репост!",
        "prob": 0.6049,
        "color": "#e74c3c" # Красный
    },
    {
        "label": "РЯДОВОЙ",
        "text": "Вчера видела, как на остановке стоял автобус номер 14. Водитель курил прямо в салоне.",
        "prob": 0.4632,
        "color": "#7f8c8d" # Темно-серый
    },
    {
        "label": "РЯДОВОЙ",
        "text": "Подскажите, пожалуйста, где в Петрозаводске можно отремонтировать старый зонт? Анонимно.",
        "prob": 0.1967,
        "color": "#bdc3c7" # Светло-серый
    }
]

# Настройка фигуры
fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 1.4) # Расширили лимит, чтобы влезли все надписи
ax.set_ylim(-0.5, len(data) - 0.5)

# Рисуем элементы
for i, item in enumerate(reversed(data)):
    # 1. Фоновая серая полоса (шкала 0-1)
    ax.barh(i, 1.0, color='#f5f5f5', edgecolor='#eeeeee', height=0.6)

    # 2. Цветная полоса вероятности
    ax.barh(i, item['prob'], color=item['color'], height=0.6, alpha=0.8)

    # 3. Текст жалобы (слева, обрезаем чуть раньше, чтобы не наезжал на цифры)
    wrapped_text = "\n".join(textwrap.wrap(item['text'], width=75))
    ax.text(0.01, i, wrapped_text, va='center', fontsize=12, color='#2c3e50', fontweight='medium')

    # 4. Значение вероятности (БОЛЬШОЕ и ЖИРНОЕ справа от бара)
    ax.text(1.02, i, f"{item['prob']:.4f}", va='center', ha='left',
            fontsize=16, fontweight='bold', color=item['color'])

    # 5. Вердикт (плашка в самом конце)
    ax.text(1.28, i, item['label'], va='center', ha='center', fontsize=11,
            fontweight='bold', color='white',
            bbox=dict(facecolor=item['color'], edgecolor='none', boxstyle='round,pad=0.6'))

# Оформление осей
ax.set_yticks([])
ax.set_xticks([0, 0.5, 1.0])
ax.set_xticklabels(['0.0 (Обычный)', '0.5 (Порог)', '1.0 (Виральный)'], fontsize=11, color='#7f8c8d')

# Вертикальная линия порога
ax.axvline(x=0.5, color='#34495e', linestyle='--', alpha=0.4, linewidth=1.5)


# Убираем лишние рамки
for spine in ['top', 'right', 'left', 'bottom']:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.savefig('model_predictions_final.png', dpi=300)
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Подготовка данных
df = pd.read_csv('/content/ptz_complaints_with_topics.csv')

topic_mapping = {
    'Тема 1 (Анализ модели)': 'Благоустройство и дороги (Снег и Ямы)',
    'Тема 2 (Анализ модели)': 'Региональная власть',
    'Тема 3 (Анализ модели)': 'Социальная сфера (Дети и Медицина)',
    'Тема 4 (Анализ модели)': 'ЖКХ и бытовые проблемы',
    'Тема 5 (Анализ модели)': 'Общественный транспорт'
}
df['topic_name'] = df['topic_name'].replace(topic_mapping)
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month

# 2. Расчет нормировки
# Считаем количество постов по Году, Месяцу и Теме
monthly_counts = df.groupby(['year', 'month', 'topic_name']).size().reset_index(name='count')

# Считаем ОБЩЕЕ количество жалоб в каждой теме за каждый ГОД
yearly_topic_totals = df.groupby(['year', 'topic_name']).size().reset_index(name='year_total')

# Соединяем таблицы и вычисляем процент (нормируем)
normalized_df = monthly_counts.merge(yearly_topic_totals, on=['year', 'topic_name'])
normalized_df['share'] = (normalized_df['count'] / normalized_df['year_total']) * 100

months_labels = ['Янв', 'Фев', 'Мар', 'Апр', 'Май', 'Июн', 'Июл', 'Авг', 'Сен', 'Окт', 'Ноя', 'Дек']
topics = normalized_df['topic_name'].unique()

# 3. Отрисовка нормированных графиков
for i, topic in enumerate(topics):
    plt.figure(figsize=(11, 5))
    sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)

    data_subset = normalized_df[normalized_df['topic_name'] == topic]

    sns.lineplot(
        data=data_subset, x='month', y='share',
        hue='year', style='year', markers=True, dashes=False, linewidth=2.5, palette='tab10'
    )

    plt.xlabel('Месяц', fontsize=12)
    plt.ylabel('Доля от годового объема темы (%)', fontsize=12)

    plt.xticks(ticks=range(1, 13), labels=months_labels)
    plt.legend(title='Год', loc='upper right')
    plt.tight_layout()

    filename = f'normalized_topic_{i+1}.png'
    plt.savefig(filename, dpi=300)
    plt.show()

In [ ]:
def debug_peak_events(df, year, month):
    print(f"\n--- ТОП СОБЫТИЙ ЗА {month}/{year} (ПО ВСЕМ ТЕМАМ) ---")
    # Фильтруем только по дате
    mask = (df['year'] == year) & (df['month'] == month)
    # Сортируем по вовлеченности (ER)
    top_posts = df[mask].sort_values(by='ER', ascending=False).head(10)

    if top_posts.empty:
        print("Данные не найдены. Проверьте, есть ли этот год/месяц в вашем df.")
        print(f"Доступные года: {df['year'].unique()}")
    else:
        for i, row in top_posts.iterrows():
            print(f"ER: {row['ER']:.2f}% | Тема: {row['topic_name']} | Текст: {str(row['text'])[:200]}...")

# Запускаем поиск истины
debug_peak_events(df, 2023, 7)
debug_peak_events(df, 2026, 3)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# 1. Подготовка данных
views_trend = df.groupby(['year', 'month'])['views'].sum().reset_index()
views_trend['date_dt'] = pd.to_datetime(views_trend['year'].astype(str) + '-' + views_trend['month'].astype(str) + '-01')
views_trend = views_trend.sort_values('date_dt')

# 2. Поиск 3 главных пиков (чтобы убрать лишние подписи)
top_peaks = views_trend.nlargest(3, 'views')

# 3. Отрисовка
plt.figure(figsize=(16, 9))
sns.set_theme(style="whitegrid")
ax = plt.gca()

# Рисуем линию (чуть толще для наглядности)
plt.plot(views_trend['date_dt'], views_trend['views'],
         marker='o', color='red', linewidth=3, markersize=10)

# --- НАСТРОЙКА ОСЕЙ ---
# Крупные деления - ГОДЫ (убираем слово "год")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# Мелкие деления - МЕСЯЦЫ
ax.xaxis.set_minor_locator(mdates.MonthLocator())

# Увеличиваем шрифты делений на осях
plt.xticks(fontsize=20)
plt.yticks(fontsize=18)

# --- АННОТАЦИЯ ПИКОВ (Крупно, без рамок, до десятых) ---
for i, row in top_peaks.iterrows():
    # Форматируем число: например, 5.2 млн
    label = f"{row['views']/1e6:.1f} млн"
    plt.annotate(label,
                 xy=(row['date_dt'], row['views']),
                 xytext=(0, 20), # Отступ выше
                 textcoords='offset points',
                 ha='center',
                 fontsize=24,   # ОЧЕНЬ КРУПНО
                 fontweight='heavy',
                 color='red')

# 4. Оформление заголовков и подписей
plt.ylabel('Сумма просмотров (в млн)', fontsize=22, labelpad=15)
plt.xlabel('Временной период', fontsize=22, labelpad=15)

# Сетка
plt.grid(which='minor', axis='x', linestyle=':', alpha=0.4)
plt.grid(which='major', axis='x', linestyle='-', alpha=0.7)

# Ограничение по Y, чтобы текст не вылезал за край
plt.ylim(0, views_trend['views'].max() * 1.25)

plt.tight_layout()
plt.savefig('views_dynamics_final.png', dpi=300)
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. Подготовка нормированных данных ---
# Считаем количество постов по Году, Месяцу и Теме
monthly_counts = df.groupby(['year', 'month', 'topic_name']).size().reset_index(name='count')
# Считаем ОБЩЕЕ количество жалоб в каждой теме за каждый ГОД
yearly_topic_totals = df.groupby(['year', 'topic_name']).size().reset_index(name='year_total')
# Соединяем и вычисляем долю в %
normalized_df = monthly_counts.merge(yearly_topic_totals, on=['year', 'topic_name'])
normalized_df['share'] = (normalized_df['count'] / normalized_df['year_total']) * 100

# --- 2. Настройка стиля и фильтрация ---
sns.set_theme(style="whitegrid")
plt.figure(figsize=(18, 11)) # Большой размер холста

# Выбираем тему (фильтруем по ключевому слову)
topic_filter = 'Благоустройство'
data_subset = normalized_df[normalized_df['topic_name'].str.contains(topic_filter)]

months_labels = ['Янв', 'Фев', 'Мар', 'Апр', 'Май', 'Июн', 'Июл', 'Авг', 'Сен', 'Окт', 'Ноя', 'Дек']

# --- 3. Отрисовка линий ---
# Увеличиваем толщину линий и размер маркеров
sns.lineplot(
    data=data_subset, x='month', y='share',
    hue='year', style='year', markers=True, dashes=False,
    linewidth=4, palette='tab10', markersize=12
)

# --- 4. АННОТАЦИИ ПИКОВ (Исправленные стили стрелок) ---

# Стиль наконечника стрелки (вынесен в переменную для удобства)
arrow_style = '-|>,head_length=1.4,head_width=0.7'

# 1. Март 2026 (Красная линия)
plt.annotate('Критическое состояние дорог\n(таяние асфальта), 2026',
             xy=(3, 31.8),         # Точка на графике
             xytext=(0.5, 52),      # Где будет текст
             fontsize=18, fontweight='bold', color='#d62728',
             arrowprops=dict(
                 arrowstyle=arrow_style,
                 lw=2, linestyle='--', color='#d62728',
                 connectionstyle="arc3,rad=0.15"
             ))

# 2. Июль 2023 (Синяя линия)
plt.annotate('Туристические сборы\nи плохая погода, 2023',
             xy=(7, 13.8),
             xytext=(8.5, 30),
             fontsize=18, fontweight='bold', color='#1f77b4',
             arrowprops=dict(
                 arrowstyle=arrow_style,
                 lw=2, linestyle='--', color='#1f77b4',
                 connectionstyle="arc3,rad=-0.15"
             ))

# 3. Декабрь 2023 (Синяя линия)
plt.annotate('Рекордные снегопады\nи гололед, 2023',
             xy=(12, 33.8),
             xytext=(8, 48),
             fontsize=18, fontweight='bold', color='#1f77b4',
             arrowprops=dict(
                 arrowstyle=arrow_style,
                 lw=2, linestyle='--', color='#1f77b4',
                 connectionstyle="arc3,rad=0.2"
             ))

# --- 5. Оформление осей и заголовков ---
plt.xlabel('Месяц', fontsize=24, labelpad=15)
plt.ylabel('Доля от годового объема темы (%)', fontsize=24, labelpad=15)

# Сетка и деления
plt.xticks(ticks=range(1, 13), labels=months_labels, fontsize=20)
plt.yticks(fontsize=20)

# Легенда
plt.legend(title='Год', fontsize=18, title_fontsize=20, loc='upper right')

# Устанавливаем лимит оси Y с запасом под надписи
plt.ylim(0, 65)

plt.tight_layout()
plt.savefig('road_peaks_final_3_points.png', dpi=300)
plt.show()